# Whisper + Pyannote Meeting Transcription Pipeline

This notebook processes the audio files from:

- `dataset/processed/train/`
- `dataset/processed/test/`

It generates **only**:

- `outputs/whisper/json/*.json`
- `outputs/whisper/transcripts/*.txt`

There are **no `train/` or `test/` subfolders inside `outputs/whisper/`**.

Pipeline:

`WAV → Faster-Whisper Turbo → Pyannote speaker diarization → Speaker A/B/C/D → JSON + TXT`


## 1. Imports and configuration

The Whisper model uses the RTX 3050 through `faster-whisper`/CTranslate2.

Pyannote uses the currently installed PyTorch runtime. In this project environment PyTorch is CPU-only, so Pyannote will automatically use CPU unless the environment is later rebuilt with CUDA-enabled PyTorch.


In [25]:
%pip -q install faster-whisper pyannote.audio python-dotenv librosa pandas

Note: you may need to restart the kernel to use updated packages.


In [26]:
from pathlib import Path
import json
import os
import shutil

import librosa
import pandas as pd
import torch
import ctranslate2

from dotenv import load_dotenv
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline


# ============================================================
# PROJECT PATHS
# ============================================================

PROCESSED_DIR = Path("dataset/processed")

WHISPER_OUTPUT_DIR = Path("outputs/whisper")
JSON_DIR = WHISPER_OUTPUT_DIR / "json"
TXT_DIR = WHISPER_OUTPUT_DIR / "transcripts"


# Create ONLY the required output folders
JSON_DIR.mkdir(parents=True, exist_ok=True)
TXT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# REMOVE OLD OUTPUT STRUCTURE
# ============================================================
# These folders were part of the previous notebook and are
# no longer used. This does NOT touch dataset/processed/train
# or dataset/processed/test.

for old_dir in [
    WHISPER_OUTPUT_DIR / "train",
    WHISPER_OUTPUT_DIR / "test"
]:
    if old_dir.exists():
        shutil.rmtree(old_dir)
        print(f"Removed old output folder: {old_dir}")


print("JSON output :", JSON_DIR)
print("TXT output  :", TXT_DIR)


JSON output : outputs/whisper/json
TXT output  : outputs/whisper/transcripts


## 2. Load Hugging Face token

The token is read from `.env`:

```text
HF_TOKEN=hf_...
```

Do not hard-code the token in the notebook.


In [27]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Add HF_TOKEN=hf_... to your .env file."
    )

print("Hugging Face token loaded:", HF_TOKEN[:5] + "*****")


Hugging Face token loaded: hf_Tu*****


## 3. Load Whisper Turbo on the RTX 3050

`faster-whisper` runs through CTranslate2, so it can use the NVIDIA GPU independently of the CPU-only PyTorch installation used for Pyannote.

The preferred RTX 3050 configuration here is:

- device: `cuda`
- compute type: `int8_float16`

If CTranslate2 cannot see CUDA, this cell stops with a clear error instead of silently running on CPU.


In [28]:
# Gracefully determine device and compute type
whisper_device = "cpu"
whisper_compute_type = "int8"

try:
    cuda_compute_types = ctranslate2.get_supported_compute_types("cuda")
    whisper_device = "cuda"
    if "int8_float16" in cuda_compute_types:
        whisper_compute_type = "int8_float16"
    elif "float16" in cuda_compute_types:
        whisper_compute_type = "float16"
    else:
        whisper_compute_type = "float32"
except Exception:
    print("CUDA not accessible, falling back to CPU.")
    pass

print(f"Loading Whisper Turbo on {whisper_device} (compute type: {whisper_compute_type})...")

whisper_model = WhisperModel(
    "turbo",
    device=whisper_device,
    device_index=0,
    compute_type=whisper_compute_type
)

print(f"Whisper Turbo loaded successfully on {whisper_device}.")


CUDA not accessible, falling back to CPU.
Loading Whisper Turbo on cpu (compute type: int8)...
Whisper Turbo loaded successfully on cpu.


## 4. Load Pyannote speaker diarization

Pyannote will identify anonymous voices such as `SPEAKER_00`, `SPEAKER_01`, etc.

They are later mapped to `Speaker A`, `Speaker B`, `Speaker C`, and so on.

Because the current project environment has CPU-only PyTorch, Pyannote uses CPU here.


In [29]:
pyannote_device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")

print("Pyannote device:", pyannote_device)

diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

if pyannote_device == "cuda":
    diarization_pipeline.to(torch.device("cuda"))

print("Pyannote pipeline loaded successfully.")


Pyannote device: mps
Pyannote pipeline loaded successfully.


## 5. Find all audio files

The files stay in their original dataset locations. The `train`/`test` split is recorded inside each JSON file for traceability, but it is **not** used to create output folders.


In [30]:
audio_files = []

train_dir = PROCESSED_DIR / "train"
test_dir = PROCESSED_DIR / "test"

for audio_file in sorted(train_dir.glob("*.wav")):
    audio_files.append(("train", audio_file))

for audio_file in sorted(test_dir.glob("*.wav")):
    audio_files.append(("test", audio_file))


print("Audio files to process:")
print()

for split, audio_file in audio_files:
    print(f"{split:5} -> {audio_file}")

if not audio_files:
    raise FileNotFoundError(
        "No WAV files found in dataset/processed/train or "
        "dataset/processed/test."
    )


Audio files to process:

train -> dataset/processed/train/ES2004a.wav
train -> dataset/processed/train/IS1009a.wav
train -> dataset/processed/train/MTG_32040.wav
train -> dataset/processed/train/MTG_32063.wav
train -> dataset/processed/train/MTG_32072.wav
train -> dataset/processed/train/MTG_32074.wav
train -> dataset/processed/train/MTG_32092.wav
test  -> dataset/processed/test/EN2002a.wav
test  -> dataset/processed/test/MTG_32179.wav
test  -> dataset/processed/test/MTG_32185.wav
test  -> dataset/processed/test/MTG_32256.wav
test  -> dataset/processed/test/MTG_32257.wav
test  -> dataset/processed/test/MTG_32322.wav
test  -> dataset/processed/test/TS3003a.wav


## 6. Speaker matching helper

Whisper and Pyannote produce time segments independently.

This helper assigns each Whisper segment to:

1. the speaker with the greatest time overlap, or
2. the nearest speaker turn when there is no direct overlap.

This avoids unnecessary `Unknown Speaker` labels caused only by tiny timestamp gaps.

The JSON records whether an assignment was based on `overlap` or `nearest`, so the result remains auditable.


In [31]:
def assign_speaker_to_segment(
    start_time,
    end_time,
    speaker_diarization
):
    overlap_scores = {}

    # --------------------------------------------------------
    # First: use actual overlap
    # --------------------------------------------------------
    for turn, _, speaker_id in speaker_diarization.itertracks(
        yield_label=True
    ):
        overlap_start = max(start_time, turn.start)
        overlap_end = min(end_time, turn.end)

        overlap = max(
            0.0,
            overlap_end - overlap_start
        )

        if overlap > 0:
            overlap_scores[speaker_id] = (
                overlap_scores.get(speaker_id, 0.0)
                + overlap
            )

    if overlap_scores:
        best_speaker = max(
            overlap_scores,
            key=overlap_scores.get
        )
        return best_speaker, "overlap"

    # --------------------------------------------------------
    # Second: use nearest speaker turn
    # --------------------------------------------------------
    midpoint = (start_time + end_time) / 2.0

    nearest_speaker = None
    nearest_distance = float("inf")

    for turn, _, speaker_id in speaker_diarization.itertracks(
        yield_label=True
    ):
        if midpoint < turn.start:
            distance = turn.start - midpoint
        elif midpoint > turn.end:
            distance = midpoint - turn.end
        else:
            distance = 0.0

        if distance < nearest_distance:
            nearest_distance = distance
            nearest_speaker = speaker_id

    if nearest_speaker is not None:
        return nearest_speaker, "nearest"

    return "UNKNOWN", "unassigned"


## 7. Process one audio file

Each file produces:

- one JSON in `outputs/whisper/json/`
- one TXT in `outputs/whisper/transcripts/`

The JSON keeps timestamps, speaker IDs, speaker labels, text, language, and assignment method.


In [32]:
def process_audio(audio_path, split_name):
    audio_path = Path(audio_path)

    print()
    print("=" * 75)
    print(f"Processing     : {audio_path.name}")
    print(f"Dataset source : {split_name}")
    print("=" * 75)

    # ========================================================
    # 1. WHISPER TRANSCRIPTION
    # ========================================================

    print("\n[1/5] Whisper Turbo transcription...")

    segments_generator, info = whisper_model.transcribe(
        str(audio_path),
        language="en",
        task="transcribe",
        beam_size=5,
        condition_on_previous_text=False,
        vad_filter=True
    )

    whisper_segments = list(segments_generator)

    print(f"Whisper segments: {len(whisper_segments)}")
    print(f"Detected language: {info.language}")
    print(
        f"Language probability: "
        f"{info.language_probability:.4f}"
    )

    # ========================================================
    # 2. LOAD AUDIO FOR PYANNOTE
    # ========================================================

    print("\n[2/5] Loading audio for speaker diarization...")

    audio, sample_rate = librosa.load(
        str(audio_path),
        sr=16000,
        mono=True
    )

    waveform = torch.from_numpy(
        audio
    ).float().unsqueeze(0)

    # ========================================================
    # 3. PYANNOTE DIARIZATION
    # ========================================================

    print("\n[3/5] Detecting speakers...")

    diarization_result = diarization_pipeline({
        "waveform": waveform,
        "sample_rate": sample_rate
    })

    speaker_diarization = (
        diarization_result.exclusive_speaker_diarization
    )

    detected_speaker_ids = sorted(
        set(
            speaker_id
            for _, _, speaker_id
            in speaker_diarization.itertracks(
                yield_label=True
            )
        )
    )

    print(
        f"Detected speakers: "
        f"{len(detected_speaker_ids)}"
    )

    print(
        "Speaker IDs:",
        detected_speaker_ids
    )

    # ========================================================
    # 4. ASSIGN SPEAKERS
    # ========================================================

    print("\n[4/5] Assigning speakers to transcript...")

    # Anonymous pyannote IDs -> friendly labels
    speaker_map = {}

    def speaker_label(speaker_id):
        if speaker_id not in speaker_map:
            letter = chr(
                ord("A") + len(speaker_map)
            )
            speaker_map[speaker_id] = (
                f"Speaker {letter}"
            )
        return speaker_map[speaker_id]

    conversation = []

    for segment in whisper_segments:
        text = segment.text.strip()

        if not text:
            continue

        start_time = float(segment.start)
        end_time = float(segment.end)

        speaker_id, assignment_method = (
            assign_speaker_to_segment(
                start_time,
                end_time,
                speaker_diarization
            )
        )

        if speaker_id == "UNKNOWN":
            label = "Unknown Speaker"
        else:
            label = speaker_label(speaker_id)

        item = {
            "start": round(start_time, 3),
            "end": round(end_time, 3),
            "speaker_id": speaker_id,
            "speaker": label,
            "assignment_method": assignment_method,
            "text": text
        }

        # Merge consecutive segments from the same speaker
        if (
            conversation
            and conversation[-1]["speaker"] == label
            and conversation[-1]["end"] <= start_time + 1.0
        ):
            conversation[-1]["text"] += " " + text
            conversation[-1]["end"] = item["end"]

        else:
            conversation.append(item)

    unknown_count = sum(
        1
        for item in conversation
        if item["speaker"] == "Unknown Speaker"
    )

    # ========================================================
    # 5. SAVE JSON + TXT
    # ========================================================

    print("\n[5/5] Saving JSON and TXT...")

    json_output = (
        JSON_DIR / f"{audio_path.stem}.json"
    )

    txt_output = (
        TXT_DIR / f"{audio_path.stem}.txt"
    )

    json_data = {
        "audio_file": audio_path.name,
        "dataset_split": split_name,
        "detected_language": info.language,
        "language_probability": round(
            float(info.language_probability),
            4
        ),
        "detected_speaker_count": len(
            detected_speaker_ids
        ),
        "detected_speakers": detected_speaker_ids,
        "speaker_mapping": speaker_map,
        "unknown_segments": unknown_count,
        "segments": conversation
    }

    with open(
        json_output,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            json_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    with open(
        txt_output,
        "w",
        encoding="utf-8"
    ) as f:
        for item in conversation:
            f.write(
                f"{item['speaker']}: "
                f"{item['text']}\n\n"
            )

    print(f"JSON saved: {json_output}")
    print(f"TXT saved : {txt_output}")
    print(f"Speakers  : {len(detected_speaker_ids)}")
    print(f"Unknown   : {unknown_count}")

    return {
        "audio_file": audio_path.name,
        "split": split_name,
        "speakers": len(detected_speaker_ids),
        "unknown": unknown_count,
        "json": str(json_output),
        "txt": str(txt_output)
    }


## 8. Process all four audio files

Run this cell once. It processes the two `train` source files and the two `test` source files, but saves all outputs directly into the two common output folders.


In [33]:
results = []

print("=" * 75)
print("STARTING ALL AUDIO PROCESSING")
print("=" * 75)

for split, audio_file in audio_files:
    # Skip files whose transcript is already generated
    txt_path = TXT_DIR / f"{audio_file.stem}.txt"
    if txt_path.exists():
        print(f"Skipping (transcript already exists): {audio_file.name}")
        continue

    try:
        result_info = process_audio(
            audio_file,
            split
        )
        results.append(result_info)

    except Exception as e:
        print()
        print(f"ERROR processing: {audio_file.name}")
        print("Error type:", type(e).__name__)
        print("Error:", e)
        print()
        continue


print()
print("=" * 75)
print("PROCESSING SUMMARY")
print("=" * 75)

if not results:
    print("No files were processed successfully.")
else:
    for item in results:
        print(
            f"{item['audio_file']:15} | "
            f"Source={item['split']:5} | "
            f"Speakers={item['speakers']} | "
            f"Unknown={item['unknown']}"
        )

print()
print("JSON outputs -> outputs/whisper/json/")
print("TXT outputs  -> outputs/whisper/transcripts/")


STARTING ALL AUDIO PROCESSING
Skipping (transcript already exists): ES2004a.wav
Skipping (transcript already exists): IS1009a.wav

Processing     : MTG_32040.wav
Dataset source : train

[1/5] Whisper Turbo transcription...
Whisper segments: 110
Detected language: en
Language probability: 1.0000

[2/5] Loading audio for speaker diarization...

[3/5] Detecting speakers...


/Users/manish/Documents/KU/.venv3.13/lib/python3.13/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  std = sequences.std(dim=-1, correction=1)


Detected speakers: 5
Speaker IDs: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03', 'SPEAKER_04']

[4/5] Assigning speakers to transcript...

[5/5] Saving JSON and TXT...
JSON saved: outputs/whisper/json/MTG_32040.json
TXT saved : outputs/whisper/transcripts/MTG_32040.txt
Speakers  : 5
Unknown   : 0

Processing     : MTG_32063.wav
Dataset source : train

[1/5] Whisper Turbo transcription...
Whisper segments: 98
Detected language: en
Language probability: 1.0000

[2/5] Loading audio for speaker diarization...

[3/5] Detecting speakers...
Detected speakers: 3
Speaker IDs: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02']

[4/5] Assigning speakers to transcript...

[5/5] Saving JSON and TXT...
JSON saved: outputs/whisper/json/MTG_32063.json
TXT saved : outputs/whisper/transcripts/MTG_32063.txt
Speakers  : 3
Unknown   : 0

Processing     : MTG_32072.wav
Dataset source : train

[1/5] Whisper Turbo transcription...
Whisper segments: 87
Detected language: en
Language probability: 1.0000

[2/5

## 9. Verify the output structure

There should be no `train` or `test` directories inside `outputs/whisper/`.


In [34]:
json_files = sorted(JSON_DIR.glob("*.json"))
txt_files = sorted(TXT_DIR.glob("*.txt"))

print("JSON files:")
for file in json_files:
    print(" ", file.name)

print("\nTXT files:")
for file in txt_files:
    print(" ", file.name)

print("\nJSON count :", len(json_files))
print("TXT count  :", len(txt_files))


JSON files:
  MTG_32040.json
  MTG_32063.json
  MTG_32072.json
  MTG_32074.json
  MTG_32092.json
  MTG_32179.json
  MTG_32185.json
  MTG_32256.json
  MTG_32257.json
  MTG_32322.json

TXT files:
  EN2002a.txt
  ES2004a.txt
  IS1009a.txt
  MTG_32040.txt
  MTG_32063.txt
  MTG_32072.txt
  MTG_32074.txt
  MTG_32092.txt
  MTG_32179.txt
  MTG_32185.txt
  MTG_32256.txt
  MTG_32257.txt
  MTG_32322.txt
  TS3003a.txt

JSON count : 10
TXT count  : 14


## 10. Quick transcript preview

This prints the first few conversation turns from each generated TXT file.


In [35]:
for txt_file in sorted(TXT_DIR.glob("*.txt")):
    print()
    print("=" * 75)
    print(txt_file.name)
    print("=" * 75)

    with open(
        txt_file,
        "r",
        encoding="utf-8"
    ) as f:
        lines = f.readlines()

    preview = "".join(lines[:20])
    print(preview)



EN2002a.txt
Speaker A: I wonder how much of the meetings is talking about the stuff at the meetings.

Speaker B: Yeah, exactly. Can I really start? Okay.

Speaker C: Should we just go around and everyone says what they've been doing? Has anybody done anything? Well, I hope so.

Speaker A: Okay. Sounds like you've done something.

Speaker C: Okay, well I've got a browser now, which, whoops, I've already got.

Speaker D: Oh, I dropped mine as well. Give it this.

Speaker C: It's supposed to be a good start.

Speaker D: Okay.

Speaker C: Yeah. So, yeah, we've got a browser which comes up automatically with the transcription box and the topics. And then when you go on the menu, you can select the summarization box, which pops up, and an audio player. And I think the search works as well, so you can pop up a search. And it loads up just the background window, so empty. And so when you start, you have to either open a particular observation or do a search and open it through that. Does that

In [36]:
# TEMPORARY RUNNING THE CODE TO CHECK WHETHER THE DEVICE CONFIG WORKS
from faster_whisper import WhisperModel
import ctranslate2

test_device = "cpu"
test_compute = "int8"
try:
    if ctranslate2.get_supported_compute_types("cuda"):
        test_device = "cuda"
        test_compute = "int8_float16" if "int8_float16" in ctranslate2.get_supported_compute_types("cuda") else "float16"
except Exception:
    pass

print(f"Loading Whisper Turbo on {test_device}...")

model = WhisperModel(
    "turbo",
    device=test_device,
    device_index=0,
    compute_type=test_compute
)

print(f"Whisper Turbo loaded successfully on {test_device}.")


Loading Whisper Turbo on cpu...
Whisper Turbo loaded successfully on cpu.
